# Data Cleaning & Exploratory Analysis - Zomato Dataset
Materi Week 3: Pandas Core (Filtering, Cleaning & Manipulation)

Fokus notebook ini:
1. Handling encoding BOM (`utf-8-sig`) dan parsing missing value terselubung
2. Identifikasi missing value eksplisit (`NaN`) vs data error implisit (angka 0 logis, 'Not rated')
3. Treatment data: imputasi Cuisines, konversi boolean, handling anomali koordinat
4. Filtering multi-kondisi dengan `.loc` dan slicing berbasis indeks integer dengan `.iloc`
5. Feature engineering teks: ekstraksi masakan utama (`Primary_Cuisine`)
6. Export dataset bersih ke format CSV

In [28]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

path = r'C:\Users\sayyi\Downloads\Zomato Restaurant Dataset.csv'

# Catatan: Gunakan encoding='utf-8-sig' agar tanda BOM di nama kolom pertama tidak error
missing_values = ['n/a', 'na', '--', '-', 'None', 'none', 'null', '?', ' ', '   ']
csv = pd.read_csv(path, encoding='utf-8-sig', na_values=missing_values)

print('Ukuran Dataset:', csv.shape)
csv.head(3)

Ukuran Dataset: (9551, 21)


,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,Average Cost for two,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenue, Poblacion, Makati City","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Makati City",121.027535,14.565443,"French, Japanese, Desserts",1100,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Makati City",121.014101,14.553708,Japanese,1200,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Mandaluyong City",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",4000,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270


In [29]:
# 1. Evaluasi Missing Value Eksplisit (NaN)
null_counts = csv.isnull().sum()
null_percent = (csv.isnull().mean() * 100).round(2)

summary_null = pd.DataFrame({
    'Total Missing': null_counts,
    'Persentase (%)': null_percent
})
display(summary_null[summary_null['Total Missing'] > 0])

,Total Missing,Persentase (%)
Cuisines,9,0.09


In [30]:
# 2. Deteksi Missing Value Terselubung / Data Error Implisit
print('=== ANOMALI / DATA IMPLISIT KOSONG ===')
print('Rating bernilai 0.0 (Belum ada review) :', (csv['Aggregate rating'] == 0).sum())
print('Rating text bernilai "Not rated"        :', (csv['Rating text'] == 'Not rated').sum())
print('Biaya makan 2 orang bernilai 0 (Gratis?):', (csv['Average Cost for two'] == 0).sum())
print('Koordinat (0, 0) di laut lepas          :', ((csv['Longitude'] == 0) & (csv['Latitude'] == 0)).sum())
print('Votes bernilai 0                        :', (csv['Votes'] == 0).sum())
print('Duplikasi ID Restoran                   :', csv.duplicated(subset=['Restaurant ID']).sum())

=== ANOMALI / DATA IMPLISIT KOSONG ===
Rating bernilai 0.0 (Belum ada review) : 2148
Rating text bernilai "Not rated"        : 2148
Biaya makan 2 orang bernilai 0 (Gratis?): 18
Koordinat (0, 0) di laut lepas          : 497
Votes bernilai 0                        : 1094
Duplikasi ID Restoran                   : 0


In [31]:
# 3. Treatment Missing Value & Standarisasi Tipe Data

# A. Isi missing Cuisines dengan 'Others' (hanya 9 baris dari 9.551)
csv['Cuisines'] = csv['Cuisines'].fillna('Others')

# B. Standarisasi kolom Yes/No menjadi boolean (True/False)
bool_cols = ['Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu']
for col in bool_cols:
    csv[col] = csv[col].map({'Yes': True, 'No': False})

print('Konversi tipe data boolean selesai:')
print(csv[bool_cols].dtypes)

Konversi tipe data boolean selesai:
Has Table booking       bool
Has Online delivery     bool
Is delivering now       bool
Switch to order menu    bool
dtype: object


In [32]:
# 4. Handling Nilai Koordinat 0.0 & Flagging Rating Valid

# Koordinat 0 diganti np.nan agar tidak merusak visualisasi geospatial
csv['Longitude'] = csv['Longitude'].replace(0.0, np.nan)
csv['Latitude'] = csv['Latitude'].replace(0.0, np.nan)

# Buat penanda apakah restoran sudah memiliki rating nyata (bukan 0.0)
csv['Is_Rated'] = csv['Aggregate rating'] > 0

print('Restoran sudah memiliki rating :', csv['Is_Rated'].sum())
print('Restoran belum memiliki rating :', (~csv['Is_Rated']).sum())

Restoran sudah memiliki rating : 7403
Restoran belum memiliki rating : 2148


In [33]:
# 5. Latihan Filtering Multi-Kondisi dengan .loc

# Kasus: Restoran dengan Rating >= 4.0, menyediakan Online Delivery, dan Table Booking
top_serviced = csv.loc[
    (csv['Aggregate rating'] >= 4.0) &
    (csv['Has Online delivery'] == True) &
    (csv['Has Table booking'] == True),
    ['Restaurant Name', 'City', 'Cuisines', 'Aggregate rating', 'Price range', 'Votes']
]
print(f'Ditemukan {len(top_serviced)} restoran dengan rating tinggi & fasilitas lengkap:')
display(top_serviced.head())

Ditemukan 118 restoran dengan rating tinggi & fasilitas lengkap:


,Restaurant Name,City,Cuisines,Aggregate rating,Price range,Votes
566,Pizza Di Rocco,Abu Dhabi,"Italian, Pizza",4.4,3,471
571,Cho Gao - Crowne Plaza Abu Dhabi,Abu Dhabi,"Thai, Japanese, Chinese, Indonesian, Vietnamese",4.4,4,246
572,Gazebo,Abu Dhabi,"Indian, North Indian, Mughlai, Biryani",4.0,3,355
578,Indian By Nature,Abu Dhabi,Indian,4.3,3,180
585,Maharaja Bhog,Dubai,"Indian, Rajasthani",4.1,2,1448


In [34]:
# 6. Latihan Slicing dengan .iloc (Integer-based Indexing)

# Mengambil 5 baris pertama (indeks 0 s/d 4) dan 4 kolom pertama (kolom 0 s/d 3)
subset_iloc = csv.iloc[0:5, 0:4]
display(subset_iloc)

,Restaurant ID,Restaurant Name,Country Code,City
0,6317637,Le Petit Souffle,162,Makati City
1,6304287,Izakaya Kikufuji,162,Makati City
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City
3,6318506,Ooma,162,Mandaluyong City
4,6314302,Sambo Kojin,162,Mandaluyong City


In [35]:
# 7. Feature Engineering Sederhana: Parsing Kolom Teks

# Hitung berapa banyak jenis masakan yang ditawarkan
csv['Cuisine_Count'] = csv['Cuisines'].apply(lambda x: len(str(x).split(',')))

# Ekstrak masakan utama (masakan pertama sebelum koma)
csv['Primary_Cuisine'] = csv['Cuisines'].apply(lambda x: str(x).split(',')[0].strip())

display(csv[['Restaurant Name', 'Cuisines', 'Primary_Cuisine', 'Cuisine_Count']].head())

,Restaurant Name,Cuisines,Primary_Cuisine,Cuisine_Count
0,Le Petit Souffle,"French, Japanese, Desserts",French,3
1,Izakaya Kikufuji,Japanese,Japanese,1
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",Seafood,4
3,Ooma,"Japanese, Sushi",Japanese,2
4,Sambo Kojin,"Japanese, Korean",Japanese,2


In [36]:
# 8. Simpan Dataset Hasil Cleaning
output_path = r'D:\Data Analyst\Python\zomato_cleaned.csv'
csv.to_csv(output_path, index=False)
print('Dataset bersih berhasil disimpan ke:', output_path)
print('Dimensi data akhir:', csv.shape)

Dataset bersih berhasil disimpan ke: D:\Data Analyst\Python\zomato_cleaned.csv
Dimensi data akhir: (9551, 24)
